# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
#import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/content/drive/MyDrive/nlp-hw2'

In [3]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,(40*0)-18=,-18
1,6+4+31=,41
2,4*11+13=,57
3,44*39*26=,44616
4,(39*42)+27=,1665


In [4]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

special_tokens = ['<pad>', '<eos>']
arithmetic_chars = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '*', '(', ')', '=']
vocab = special_tokens + arithmetic_chars

char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
id_to_char = {idx: ch for idx, ch in enumerate(vocab)}

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

Vocab size18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# Write your code here
pad_id = char_to_id['<pad>']
eos_id = char_to_id['<eos>']

srcs = df_train['src'].values
tgts = df_train['tgt'].values

char_id_list = []
label_id_list = []

for s,t in zip(srcs,tgts):
    c_ids = [char_to_id[c] for c in s]
    prefix_len = len(s) - len(t)-1
    l_ids = [pad_id] * prefix_len + [char_to_id[c] for c in t] + [eos_id]
    char_id_list.append(c_ids)
    label_id_list.append(l_ids)

df_train['char_id_list'] = char_id_list
df_train['label_id_list'] = label_id_list

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,(40*0)-18=-18,-18,13,"[15, 6, 2, 14, 2, 16, 13, 3, 10, 17, 13, 3, 10...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 3, 10, 1]"
1,6+4+31=41,41,9,"[8, 12, 6, 12, 5, 3, 17, 6, 3, 1]","[0, 0, 0, 0, 0, 0, 0, 6, 3, 1]"
2,4*11+13=57,57,10,"[6, 14, 3, 3, 12, 3, 5, 17, 7, 9, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 7, 9, 1]"
3,44*39*26=44616,44616,14,"[6, 6, 14, 5, 11, 14, 4, 8, 17, 6, 6, 8, 3, 8, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 8, 3, 8, 1]"
4,(39*42)+27=1665,1665,15,"[15, 5, 11, 14, 6, 4, 16, 12, 4, 9, 17, 3, 8, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 8, 8, 7, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

這六個參數是深度學習（特別是 NLP 與 RNN 序列模型）中最核心的「遊戲規則與硬體規格」。
###為什麼叫「超參數（Hyperparameter）」？

一般參數（Parameters / Weights）：模型自己「邊學邊調整」的神經元權重。
超參數（Hyperparameters）：在訓練開始前，由人類工程師手動指定的環境與架構設定。模型自己不會改動它們，但它們決定了模型「學多快、記性多好、怎麼讀書」。

###batch_size = 64 （單批資料量 / 做題節奏）
    1. GPU 意義：GPU（尤其是你的 A100）擅長平行矩陣運算。把 64 題打包成一個大矩陣丟進去，A100 可以在一瞬間同時算完，大幅提高訓練吞吐量。
    2. 權衡（Trade-off）：
        太小：GPU 算力被閒置，訓練變慢。
        太大：顯存會被塞爆（CUDA Out of Memory），且模型容易失去隨機泛化能力。

###epochs = 2（訓練輪數 / 題庫刷題次數）
    本次資料量高達 100 萬筆，資料非常充足。在 A100 上跑 2 個 Epoch（總共看過 200 萬次題目）

###embed_dim = 256 （字元嵌入維度 / 符號特徵長度）
    用 256 個面向（維度）來描摹一個符號

###hidden_dim = 256 （隱藏層維度 / LSTM 的「腦容量」）
    算式是一連串依序輸入的字元，例如 (1+2)*3=。
    當模型讀到 * 時，它必須在腦中記住「前面括號裡算出來是 3」。這個保留歷史記憶的載體就是 LSTM 的 Hidden State（隱藏狀態）。
    hidden_dim = 256 代表模型在每讀一個字元時，都用一個長度 256 的向量來記錄「當前運算狀態與歷史脈絡」。

###lr = 0.001（Learning Rate，學習率 / 下山步伐）
    1.模型在算完損失（Loss）後，要調整自己的大腦神經元權重。lr 決定了每一次調整的步伐大小： W新 = W舊- lr * 梯度
    2.權衡（Trade-off）：
        步子太大（如 0.1）：扯到蛋，直接踩過最低點，在山谷兩側劇烈震盪，Loss 無法收斂。
        步子太小（如 0.000001）：爬得極慢，幾天幾夜都學不會。
        0.001 是深度學習優化器（特別是 Adam）最經典、最穩定的黃金預設值。

###grad_clip = 1（梯度裁剪 / 防止爆炸的保險絲）
1. **背景**：RNN 在時間軸上一層一層反向傳遞梯度（BPTT）。當算式很長時，連乘效應容易導致梯度指數型暴增，稱為「梯度爆炸（Gradient Explosion）」。
2. **本程式機制 (`clip_grad_value_`)**：
   - 規定梯度張量中的**「每一個純量數值（Scalar）」**若超過 1 就強制截斷在 `[-1, 1]` 區間內（底層做 `clamp(-1, 1)`）。
   - ⚠️ **注意細節**：因為是每個分量各自截斷，**可能會稍微扭曲梯度的原始下山方向**；
   - 💡 **延伸對比**：若改用 `clip_grad_norm_`，才是「檢查向量總長度並等比例縮小」，那樣才能「100% 保持方向不變」。

In [ ]:
batch_size = 64 
epochs = 2
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
# ==================== TODO 3: 資料打包類別與整理函式 ====================

# 【倉儲取貨員】：定義如何從 100 萬筆資料中，單筆單筆抽出 (題目 x, 標籤 y)
#class Dataset(torch.utils.data.Dataset): 代表繼承（Inheritance）
class Dataset(torch.utils.data.Dataset): #torch.utils.data.Dataset: PyTorch 官方制定的標準資料集規範
    def __init__(self, sequences): #在 Python 的類別Class函式中，第一個參數永遠是 self代表「被建立出來的這個物件實體（自己）」。
        self.sequences = sequences

    def __len__(self):
        # 回傳總資料筆數 (100萬)，供 DataLoader 與進度條計算總步數
        return len(self.sequences)

    def __getitem__(self, index):
        # 根據給定索引，抽出第 index 筆的 char_id_list (x) 與 label_id_list (y)
        x = self.sequences.iloc[index]['char_id_list']
        y = self.sequences.iloc[index]['label_id_list']
        return x, y

# 【裝箱打包員】：DataLoader 每抓取 64 筆資料後，透過此函式進行對齊與打包
def collate_fn(batch):
    '''
    0. 參數 batch 是誰傳進來的？它裡面到底長怎樣？
    誰傳進來的：是後面的 DataLoader 自動傳進來的。
    怎麼產生的：
    當我們設定 batch_size = 64，DataLoader 就會去呼叫剛才的 Dataset.__getitem__ 整整 64 次。
    每一次 __getitem__ 都會回傳一個元組 (x, y)（題目與答案）。
    真實結構： 所以傳進 collate_fn 的 batch，本質上是一個裝了 64 個元組的 Python List：
    python
    batch = [
        (x_0, y_0),    # 第 1 題的題目與答案
        (x_1, y_1),    # 第 2 題的題目與答案
        ...
        (x_63, y_63)   # 第 64 題的題目與答案
    ]
    '''
    # 1. 將 batch (64 個 tuple) 拆分成 x 列表與 y 列表，並轉為 PyTorch 張量
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    '''
    1. batch_x = [torch.tensor(data[0]) for data in batch] 在幹嘛？
    這是 Python 的列表生成式（List Comprehension）：
    迴圈走訪 batch 裡的每一個元素 data（也就是每一對 (x, y)）。
    data[0]：拿出第 0 個位置的東西，也就是 題目 x（如 [3, 12, 3, 17, 4, 1]）。
    torch.tensor(...)：原本從 Pandas 出來的是一般 Python List，這裡把它轉成 PyTorch 張量（Tensor），GPU 才能做矩陣計算。
    （下一行的 data[1] 就是拿出 答案標籤 y）。
    '''
    # 2. 記錄這 64 題每一題未填充前的「真實原始長度」
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    '''
    2.為什麼要記長度？
    在這 64 題中，有的題目很短（如 1+1=2，長度 6），有的很長（如 (12+34)*56=2576，長度 16）。
    等一下短的算式會被塞滿 0（<pad>）補平成 16。
    如果沒有提前把「原本的長度（6 和 16）」記錄下來，後面的神經網路就會分不清哪些是真正的算式內容、哪些只是後面補上的空白廢話！
    為什麼是 LongTensor？
    在 PyTorch 中，LongTensor 代表 64 位元整數（int64）。
    深度學習慣例：權重浮點數用 FloatTensor，維度長度、索引、Token ID 全部用 LongTensor。
    '''

    # 3. 動態填充 (Padding)：因為每題長短不一，GPU 無法平行計算鋸齒狀矩陣
    #    pad_sequence 會以該批次最長長度為基準，短的後方自動補 0 (<pad>)
    #    batch_first=True 使輸出形狀為 (64, max_seq_len)
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    '''
    3. GPU 只能運算平整的矩陣，沒辦法處理鋸齒狀陣列：
    補齊前（鋸齒狀，GPU 無法計算）：
    第 1 題: [1, 2, 3]        (長度 3)
    第 2 題: [4, 5, 6, 7, 8]  (長度 5)
    pad_sequence 自動補齊後（整齊長方形矩陣）：
    第 1 題: [1, 2, 3, 0, 0]  (自動在後面補兩個 0: <pad>)
    第 2 題: [4, 5, 6, 7, 8]  (維持長度 5)
    padding_value = char_to_id['<pad>']：短的題目後面用 0（<pad> 的 ID）補齊。
    batch_first=True：
    若設為 True，產出的張量形狀為： 
    (batch_size,max_seq_len)=(64,16)
    (batch_size,max_seq_len)=(64,16) 代表第 1 維是「第幾題」、第 2 維是「第幾個字」，最符合人類思考習慣！

    batch_first=True 到底是什麼意思？True 和 False 的差別？
    這牽涉到多維矩陣在電腦裡的**「維度排隊順序」**！
    我們現在有 64 題算式（Batch），每題最長補到 16 個字（Seq Len）：
    情況 A：batch_first=True（以 Batch 為先，最符合人類直覺！）
    矩陣的形狀為： 
    (64,16)=(Batch Size,Sequence Length)
    (64,16)=(Batch Size,Sequence Length)
    怎麼讀：
    第 1 個維度（列）：代表「第幾題」（第 0 題到第 63 題）。
    第 2 個維度（行）：代表「第幾個字」（第 0 字到第 15 字）。
    查資料時：tensor[5, 2] 代表 「第 5 題的第 2 個字」。
    👉 完全符合我們看一張考卷的習慣（一行就是一題完整算式）！
    情況 B：batch_first=False（以時間步為先）
    矩陣的形狀會被顛倒旋轉成： 
    (16,64)=(Sequence Length,Batch Size)
    (16,64)=(Sequence Length,Batch Size)

    怎麼讀：
    第 1 個維度代表「時間步（第幾個字）」，第 2 個維度才代表「第幾題」。
    查資料時：tensor[2, 5] 代表 「在第 2 個時間點，第 5 題的字是什麼」。
    👉 反人類直覺！但在早期 RNN 底層 C 語言計算時稍微快一點點，所以 PyTorch 官方早期預設是 False。`
    '''
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens
# ====================================================================

In [ ]:
# ==================== TODO 3: 實體化訓練資料集 (ds_train) ====================
# 1. df_train[['char_id_list', 'label_id_list']]: 
#    使用雙層中括號，從原始 100 萬筆 DataFrame 中精準「切出」訓練真正需要的這兩欄子表格
# 2. Dataset(...): 
#    呼叫前面定義的 Dataset 類別，將子表格傳入 __init__ 封裝成 PyTorch 認可的標準資料集物件
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])
# ==============================================================================
'''
為什麼是「雙層中括號 [[ ... ]]」？（Pandas 必考經典語法）
很多人看到 df_train[['char_id_list', 'label_id_list']] 會納悶：為什麼要有兩層括號？

    我們把它拆開來看：
    外層的 [ ]：是 Pandas 的選取語法（df[...]）。
    內層的 [ ]：是一個標準的 Python List 清單：['char_id_list', 'label_id_list']。

    單層 vs 雙層的差別：
    寫法	語法意義	傳回型態	範例
    單層 df['src']	只選取單一欄位	Series（一維陣列）	只有單純的一條直行資料
    雙層 df[['col1', 'col2']]	選取多個欄位清單	DataFrame（二維子表格）	依然是一張具有欄名和列號的小表格
'''

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)# Write your code here
'''
DataLoader：它是「自動化發牌與運送卡車」！
前面的 ds_train（Dataset）只是倉庫裡安靜躺著的 100 萬題算式。
DataLoader 是 PyTorch 的核心工具，負責把這些算式自動洗牌、分批（64 題一組）、送去包裝（collate_fn）、最後一箱一箱送到 GPU 的嘴巴前！
它括號裡的 4 個參數，各自扮演著無可取代的任務：
    1. ds_train（資料來源庫）
    意思是：告訴卡車「要去哪一個倉庫載貨？」。
    我們把剛剛建立好的 ds_train = Dataset(...) 傳進去，卡車需要拿資料時，就會自動去呼叫 ds_train 的 __len__ 和 __getitem__。

    2. batch_size=batch_size（一次載幾筆？這裡 = 64）
    意思是：一次湊滿 64 題才出貨。
    卡車每次會從 ds_train 隨機抽 64 個樣本，湊成一組。

    3. shuffle=True（隨機洗牌，🔥 經典面試與報告考點）
    意思是：「每一輪（Epoch）開始前，要不要把 100 萬題的順序徹底重新洗牌？」
    為什麼訓練時一定要設為 True？（如果不洗牌會怎樣？）
    防止模型死記順序：如果資料集剛好前面 10 萬筆全都是加法、中間全都是減法、後面全都是乘法，如果不洗牌，模型學完乘法就會把加法忘光光（災難性遺忘 Catastrophic Forgetting）！
    確保隨機梯度下降（SGD）的數學有效性：隨機打亂能保證每一次算出來的梯度方向具有隨機性（代表整體資料分佈），幫助模型逃離局部最差的死胡同（Local Minima）。
    (小知識：如果是驗證集或測試集，shuffle 通常會設為 False，因為考試不用打亂順序，但訓練集務必設為 True)。

    4. collate_fn=collate_fn（指定包裝工人）
    意思是：當卡車抽完 64 題之後，要把這 64 題交給誰來處理？
    我們把上一格寫好的 def collate_fn(batch): 傳給它。
    卡車抽完 64 題，立刻交給 collate_fn 去塞保麗龍補 <pad>，將這 64 題對齊成形狀 (64, 16) 的長方形矩陣！
'''


# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    '''
    "self" : 在 Java/C# 中，你在不同 Class 裡寫 this，它指向的都是「當前這個 Class 被 新增出來的物件實體」，對吧？
            Python 的 self 也是一模一樣的道理！
    '''
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer
        # 零件 1：字元嵌入層
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        '''
        1. torch.nn.Embedding（字典查表器）
            num_embeddings=vocab_size (18)：字典裡總共只有 18 個字元。
            embedding_dim=256：把每個字元轉成 256 維的浮點數向量。
            padding_idx=char_to_id['<pad>'] (0)：
            關鍵細節：告訴 PyTorch「0 號字元是沒意義的填充符號」，它的 Embedding 向量永遠強制保持為全 0，反向傳播時不會更新它的梯度！
        '''
        # 零件 2：兩層循環神經網路 (RNN)
        self.rnn_layer1 = torch.nn.RNN(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)

        self.rnn_layer2 = torch.nn.RNN(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        '''
        2. self.rnn_layer1 與 self.rnn_layer2（雙層記憶網絡）
            第一層：input_size=embed_dim (256)，吃進字元向量，吐出長度 256 的隱藏狀態 ℎ𝑡
            第二層：input_size=hidden_dim (256)，把第一層的輸出當作輸入，再疊一層記憶網絡（抽取更複雜的運算先後順序）。
            batch_first=True：剛才聊過的老朋友，代表輸入矩陣的第 1 維是 64 筆 Batch。

        補充參數: 
        input_size（吃進來的向量有多長？）：
            rnn_layer1 的輸入是 Embedding 的輸出，所以它的 input_size = embed_dim = 256。
            rnn_layer2 的輸入是 rnn_layer1 吐出來的輸出，所以它的 input_size 必須等於前一層的 hidden_size = 256！
            hidden_size（吐出去的記憶向量有多長？）：
            兩層都設為 hidden_dim = 256，代表每一層內部都維持 256 個記憶槽位。
            為什麼要疊兩層？
            第一層 RNN 像「小學部」：負責辨識最基本的字元順序（例如 1 後面跟著 +）。
            第二層 RNN 像「國中部」：站在第一層的肩膀上，抽取出更複雜的數學結構（例如「這個括號什麼時候該結算？」）。
            
        batch_first=True 在 nn.RNN 裡到底在做什麼？
            這是一個天大的防呆關鍵！
            在 PyTorch 的官方設計裡，torch.nn.RNN 的預設值居然是 batch_first=False！
            如果是預設的 False：RNN 會強迫要求你餵給它的矩陣形狀是：
            (seq_len,batch_size,embed_dim)=(16,64,256)
            (seq_len,batch_size,embed_dim)=(16,64,256)
            （時間步長排在最前面）。
            為什麼我們一定要明確寫 batch_first=True？
            因為我們剛才在 collate_fn 裡，已經用 batch_first=True 把資料打包成長這樣了：
            (batch_size,seq_len,embed_dim)=(64,16,256)
            (batch_size,seq_len,embed_dim)=(64,16,256)
            👉 如果在宣告 nn.RNN 時沒有加 batch_first=True，PyTorch 會把 64 誤當成時間步、把 16 誤當成批次，整條生產線維度直接撞車報錯！
            所以這裡寫 batch_first=True，就是告訴 RNN：「聽好！我傳進來的第 1 個維度是 64 筆 Batch，不要搞反了！」
        '''
        # 零件 3：輸出線性層 (分類器)
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),# (256 -> 256)
                                          torch.nn.ReLU(),# [256 個各個標量]如果小於 0(負數)，會直接砍成 0
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size)) # (256 -> 18)
        '''
        3. self.linear（從 256 維向量猜測 18 個字元誰最有可能）
            RNN 吐出來的是一個 256 維的抽象記憶向量。
            最後必須透過 torch.nn.Linear，把維度從 256 維 → 18 維（vocab_size）。
            這 18 個數字就是對應 18 個字元的**「信心分數（Logits）」**，分數最高的那個字，就是模型預測的答案！
            Linear 是什麼？（維度投影機）
                RNN 讀完算式後，腦袋裡留下的最後狀態是一個 256 維的浮點數向量。 
                但人類只想要知道：下一個字是 18 個字元中的哪一個？ 
                Linear（線性全連接層）的數學本質就是矩陣乘法 
                𝑦=𝑥𝑊𝑇+𝑏y=xWT+b。
                最後一個 Linear 把向量維度從 256 降維投影到 18（vocab_size）。
                這 18 個數字就是 18 個字元的**「信心得分（Logits）」**！
            補充: 
            1.為什麼一定要把 ReLU 夾在兩個 Linear 之間？
            夾在中間的 ReLU，就像把直線折出一個彎角（非線性開關），打破了矩陣相乘的結合律，讓神經網路真正具備了**「非線性邏輯推演的大腦能力」**！
            否則如果不夾 ReLU，連續寫兩個 Linear兩個矩陣相乘本質上只是一個新的矩陣，數學上完全等價於「只寫一個 Linear」！
            2.激活函數（如 ReLU）跟 Softmax 是完全不同的東西嗎？
                是的，完全不同！

                ReLU（神經元的開關）：
                用在神經網路的中間層（隱藏層）。
                每個神經元自己管自己，彼此不干涉。
                Softmax（計票員 / 機率分母）：
                只用在神經網路的最後一層（輸出層）。
                它會把大家的分數全部看過一遍，算總和，把分數轉換成 「大家加起來剛好 100% 的機率分佈」。
        '''
    # forward 隸屬於 PyTorch 的父類別 torch.nn.Module
    # 因此是繼承並「複寫（Override）」來的方法！PyTorch 官方規定每個模型都必須實作自己的 forward前向傳播。
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    '''
    本範例把真正的計算過程寫在底下的 encoder() 裡面，然後讓 forward 當「前台接待員」
    只要有人叫 forward，他就直接把工作原封不動轉交給後面的 encoder 去算！
    通常會把encoder 裡的內容直接寫到forward()
    語法問題: 
        須遵守父類別定下來的遊戲規則：外界不能直接敲 forward 的門，必須遵守 PyTorch 的生命週期，在外面寫 model(x)，由 PyTorch 的總管（__call__）按部就班幫你啟動！
    '''
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        # 步驟 1：字元查表，向量化
        batch_x = self.embedding(batch_x)
        # 步驟 2：壓縮打包（跳過 pad 的黑科技）
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        # 步驟 3：通過兩層 RNN
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        '''
        (語法問題) batch_x, _ = ...為什麼等號左邊有兩樣東西？_ 是什麼？（Python 元組拆包 Tuple Unpacking）
            PyTorch 的 RNN 跑完後，一定會同時回傳 兩個結果：
                1.output：每一個時間點（每個字）的輸出。
                2.h_n：最後一個時間點的記憶（Hidden State）。

            在 Python 裡，當一個函式回傳兩個東西時，你可以用逗號一次接收： "變數1, 變數2 = 函式()"
            這裡我們只需要第 1 個 output（存回 batch_x），第 2 個 h_n 我們用不到。
            在 Python 慣例中，_（單一底線）就是一個合法的變數名稱，專門用來代表「這個垃圾我不要，丟進垃圾桶」！

        (語法問題) self.rnn_layer1(batch_x)  # 👈 為什麼變數後面可以加括號？這不是方法啊！
            Python 允許「把物件當成函式呼叫」！
            在 Python 裡，只要一個類別有實作 def __call__(self): 這個特殊方法，那麼這個物件的實體，就可以直接被當成函式掛上 () 執行！
            簡單來說Python 模糊了「資料（變數）」與「行為（方法）」的界線。
            變數不是變成了方法，而是方法本身就是一種變數，而變數只要裝上了 __call__ 晶片，就能展現出方法的行為。

        '''
         # 步驟 4：解壓縮，還原成長方形矩陣
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        # 步驟 5：線性分類，預測 18 個字元的機率
        batch_x = self.linear(batch_x)

        return batch_x

    
    # 神經網路最迷人的**「自回歸生成器（generator）」**！
    #💡 這段程式碼的含金量極高：
    #現在全世界的 ChatGPT、Claude、LLaMA 在生成文字時，底層的生成邏輯跟這段「一模一樣」——都是自己產出一個字，再把這個字塞回輸入，接著猜下一個字！
    def generator(self, start_char, max_len=200): #自訂的方法
        # 步驟 1：把人類題目（如 '1+1='）查字典轉成數字 ID 清單
        char_list = [char_to_id[c] for c in start_char]

        next_char = None
        # 步驟 2：開始一個字一個字生成，直到遇到 <eos> 或達到長度上限
        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            # Input the tensor to the embedding layer, RNN layers, linear respectively
            
            # 2-1. 轉成 Tensor，並自動配對到模型所在的 GPU/CPU 裝置
            input_tensor = torch.tensor(
                [char_list],                                # 參數 1: 資料本身
                dtype=torch.long,                           # 參數 2: 資料型態
                device=next(self.parameters()).device       # 參數 3: 存放在哪個硬體
                )
            '''
            ① [char_list]（為什麼要再包一層外括號？）
                char_list 原本是一維清單：[3, 12, 3, 17]（長度 4）。
                如果直接傳，形狀只有 (4,)。
                但神經網路嚴格規定：第 1 維必須是 Batch Size！
                在外面加一層 [ ]，變成 [[3, 12, 3, 17]]，形狀瞬間變成 (1, 4)（代表 Batch=1，字數=4），符合神經網路規格！
            ② dtype=torch.long
                剛才聊過的老朋友：整數 ID 一律用 64-bit 整數（torch.long），不能有小數點。
            ③ device=next(self.parameters()).device
                自動偵測目前模型住在哪裡（CPU 還是 GPU A100）。
                self.parameters() 是模型所有神經元權重的清單；next(...).device 會抓出第一個權重住在 CPU 還是 cuda:0。
                這樣新做出來的 input_tensor 就會自動被送到跟模型同一個地方，絕不會發生「資料在 CPU、模型在 GPU」打架的慘劇！
            
            2.1 torch.tensor() 到底做了什麼？最後回傳什麼？
                它在背後做的事情（搬運工 + 封裝工）：
                申請記憶體：向作業系統（或 GPU 顯存）申請一塊連續的記憶體空間。
                複製資料：把傳進去的 Python 清單（如 [[3, 12, 3, 17]]）的所有數字，用 C++ / CUDA 底層的高速格式拷貝進去。
                物件包裝：打包成一個功能強大的 torch.Tensor 物件。
                最後回傳什麼？
                它回傳的是一個 PyTorch 張量物件（torch.Tensor），印出來長這樣：
                    python
                    tensor([[ 3, 12,  3, 17]], device='cuda:0', dtype=torch.int64)
                它已經不再是普通 Python List 了，它現在是一塊隨時可以用 GPU 並行計算矩陣乘法的高效能張量！
            '''
            # 2-2. 走過神經網絡生產線
            out = self.embedding(input_tensor)
            out, _ = self.rnn_layer1(out)
            out, _ = self.rnn_layer2(out)
            out = self.linear(out)
            # 2-3. 關鍵切片：只拿「最後一個字元」對下一個字的預測
            y = out[0, -1, :] # Obtain the next token prediction y
            '''
            1.當我們把 '1+1='（長度 4）丟進模型後，out 的三維形狀是： 
                𝑜𝑢𝑡的形狀=(1,4,18)=(第幾題,第幾個字,18 個候選字的得分)
                out 的形狀=(1,4,18)=(第幾題,第幾個字,18 個候選字的得分)

                第 1 維 (0):        只看第 0 題（因為現在只有 1 題在考試）
                第 2 維 (4 個字元):  
                    位置 0 (讀了 '1') ➔ 模型預測下一個字的 18 個分數
                    位置 1 (讀了 '+') ➔ 模型預測下一個字的 18 個分數
                    位置 2 (讀了 '1') ➔ 模型預測下一個字的 18 個分數
                    位置 3 (讀了 '=') ➔ 模型預測下一個字的 18 個分數 👈 【我們只想要這一個！】
                第 3 維 (:):        把該位置對 18 個字的得分全部拿出來！

            2.為什麼第 2 維是 -1？是不是因為等號的 ID 是 -1？
            因為在 Python 中： −1永遠代表𝑙𝑒𝑛()−1（倒數最後一個）−1永遠代表len()−1（倒數最後一個）
                寫法 A（老實人寫法，每次都要算長度）：
                y = out[0, len(char_list) - 1, :]
                # 寫法 B（Python 的天才優雅寫法，直接用 -1）：
                y = out[0, -1, :]
                因為在 while 迴圈中，算式的長度會一直變長（4 個字 ➔ 5 個字 ➔ 6 個字）。
                我們每一次都只想要「剛放進去的最新那一個字元」後面所對應的預測！
                用 -1 不用每次呼叫 len()，永遠精準抓到最新的那一步！
            '''

            # 2-4. 挑出 18 個字元中分數最高的那個字
            next_char = torch.argmax(y).item() # Use argmax function to get the next token prediction
            '''
            torch.argmax(y)：
                作用：找出這 18 個分數中，「最高分的是哪一個編號」！
                假設第 4 號候選字（代表字元 '2'）的分數最高（比如得分是 8.5，其他都是 1.0），它就會找出編號 4，吐出 tensor(4)。
            .item()：
                把 PyTorch 的張量包裝 tensor(4) 脫掉，還原成 Python 的原生純整數 4，這樣才能 append 進 Python 清單裡。
            '''

            # 2-5. 踩煞車：如果猜出 <eos>，代表算完了，立刻結束！
            if next_char == char_to_id['<eos>']:
                break

            # 2-6. 把猜出來的新字塞進清單，下一輪繼續猜
            char_list.append(next_char)

        # 步驟 3：查反向字典，把 ID 清單還原成人類看得懂的字元
        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')# Write your code here. Specify a device (cuda or cpu)

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)
'''
1. torch.manual_seed(2)（固定隨機種子 / 防靈異現象）
    意思是：把所有的「亂數骰子」固定在第 2 號模式。
    為什麼要寫：
    神經網路剛誕生時，裡面的權重都是「隨機猜測」亂數初始化的。
    如果不固定種子，你每一次執行，模型初始化的數值都長不一樣，跑出來的準確率可能一下 80%、一下 60%，很難 Debug。
    寫上這行，保證每一次跑程式，隨機數順序 100% 一模一樣，實驗結果完全可重現（Reproducible）！
2. device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
   「三元運算子（Ternary Operator）/ 單行條件式」: [條件成立時的值]if[條件判斷]else[條件不成立時的值]
    意思是：「有顯卡用顯卡，沒顯卡用 CPU（自動切換）」。
    這是 Python 的經典單行 if-else：
    torch.cuda.is_available()：先問電腦「現在有沒有 NVIDIA 顯卡能用？」
    如果有 ➔ 就用 'cuda'（你的 Colab A100 GPU）。
    如果沒有 ➔ 自動降級用 'cpu'。
    這樣不管把程式搬去哪台電腦，都不會因為找不到顯卡而報錯崩潰！
3. model = CharRNN(vocab_size, embed_dim, hidden_dim)
    意思是：「按照藍圖，把這隻神經網路大腦真正生出來（實體化）！」
    它會呼叫剛才我們聊完的 CharRNN.__init__，傳入三項重要規格：
    vocab_size = 18（字典只有 18 個字）
    embed_dim = 256（字元特徵長度 256）
    hidden_dim = 256（LSTM 記憶容量 256）
    執行完這行，一隻全新的、熱騰騰的神經網路大腦 model 就正式誕生在記憶體裡了！
'''

In [ ]:
#在 PyTorch 慣例中，大家習慣把損失函數變數取名為 criterion。
#CrossEntropyLoss（交叉熵損失）Softmax 就藏在它的肚子裡，它會自動幫我們做機率運算
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])# Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
'''
🔥 最核心的參數：ignore_index=char_to_id['<pad>']（也就是 0）
    意思是：考官遇到 0 分直接略過，不扣分也不算分！
    還記得我們在 TODO 2 做了什麼嗎？
    題目部分（如 1+1=）我們故意把標籤全部填上了 0（<pad>）；
    後面為了對齊塞的保麗龍也是 0。
    設定 ignore_index=0，就是嚴正警告考官：
    👉 「看到標籤是 0 的地方，通通閉上眼睛！只准對等號後面的真正答案（如 '2'）與 '<eos>' 計算 Loss，猜錯題目不准懲罰！」
'''

#optimizer（負責調整神經元大腦的隨隊教練：優化器）
optimizer = torch.optim.Adam(model.parameters(), lr=lr)# Write your code here. Use Adam or AdamW for Optimizer
'''
    Adam 優化器（深度學習最經典、最聰明的自適應優化器）。
    它拿著兩樣東西工作：
        1.model.parameters()（學生的所有大腦神經元名冊）：
        把我們剛才 CharRNN 裡的所有 Embedding、RNN、Linear 的幾十萬個權重矩陣全部交給教練保管。
        2.lr=lr（調整步伐，0.001）：
        我們今天花了很多時間聊的經典步長，規定教練每次修正神經元時的幅度！
'''


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [14]:
drive_checkpoint_dir = '/content/drive/MyDrive/nlp-hw2'
drive_checkpoint_path = os.path.join(drive_checkpoint_dir, 'checkpoint.pt')
local_checkpoint_path = 'checkpoint.pt'

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device) #把整個神經網路大腦搬到顯卡上，享受高速運算。
model.train() #切換為「訓練模式」（通知所有神經網路層現在正在上課訓練中）
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    '''
    tqdm 是內建的嗎？ 
        ❌ 不是 Python 內建！ 它是一個極為著名的第三方開源套件（在最上方有寫 from tqdm import tqdm）。
        它的名字來自阿拉伯語 taqaddum（意思是「進展、進步」）。
    它的功能是什麼？
        「全自動進度條製造機」！你只要把任何可迭代的東西包進 tqdm()，它就會在終端機畫出一條動態進度條，還會自動估算每秒跑幾題（it/s）、還要剩幾分鐘（ETA）。 
        desc=f"Train epoch {epoch}"：
        是的，就是字串！ desc 是 Description（描述文字） 的縮寫。
        它就是進度條最左邊要印出來的標題名稱，例如：Train epoch 1: 45% [████░░░░]。
    '''

    #在 Python 中，in 後面可以放任何「能一個一個拿出東西的容器（可迭代物件 Iterable）」！
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar: 
        '''
        bar 就是我們前面做好的 dl_train（DataLoader），只是被套上了進度條外皮。
        batch_x：64 題算式 ID
        batch_y：64 題標籤 ID
        batch_x_lens：64 題的真實題目長度
        batch_y_lens：64 題的真實答案長度
        '''
        # Write your code here
        # Clear the gradient
        # 步驟 1：梯度清零
        optimizer.zero_grad()
        '''
        每回合都要歸零!!
        PyTorch 奇葩特性：PyTorch 的梯度預設是**「累加」**的（如果上一批算出來是 3，下一批算出來是 2，它會直接加在一起變成 5）。
        '''

        # 步驟 2：前向傳播（做題目，猜答案）
        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        # 步驟 3：計算損失（對答案，打分數）
        loss = criterion(batch_pred_y.view(-1, vocab_size), batch_y.to(device).view(-1))
        '''
        ① criterion 是誰？
            在 Cell 18 宣告的 CrossEntropyLoss（交叉熵考官）。它有 __call__，所以可以像函式一樣加括號執行。
        ② 兩個參數分別是什麼？
            參數 1（模型考卷）：batch_pred_y（模型預測的 18 個字元得分）。
            參數 2（標準答案）：batch_y（真實的字元標籤）。
            batch_y.to(device)：答案原本在 CPU，先搬去 GPU 才能跟模型打分數。
        ③ 🔥 什麼是 .view()？（PyTorch 的矩陣重塑 Reshape）
            考官 CrossEntropyLoss 規定：他不收三維矩陣，他只收「二維分數 vs 一維答案」！
            batch_pred_y.view(-1, vocab_size)：
            原本是 (64 題, 16 字, 18 分數) 的三維立體。
            -1 代表**「懶人自動推算」**（64×16=102464×16=1024）。
            瞬間被**「壓扁成一張平整的二維考卷」**：(1024 個字, 18 個分數)！
            batch_y.view(-1)：
            原本是 (64 題, 16 字) 的二維答案。
            瞬間被**「拉直成一長條一維標準答案」**：(1024 個標籤)！
        ④ 回傳的 loss 是什麼？
        回傳一個 0 維的純量張量（Scalar Tensor），例如 tensor(2.4513)。它代表這 1024 個字算下來的平均錯誤率！
        '''

        # Back propagation
        # 步驟 4：反向傳播（算出斜率梯度）+ 梯度裁剪（防爆保險絲）
        loss.backward()
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping
        '''
        (注意：PyTorch 函式名稱最後面如果帶有底線 _，代表 In-place 原地修改，意思是它直接在記憶體裡把數值改掉，不額外浪費記憶體建立新物件！)
        '''

        # Write your code here
        # Optimize parameters in the model
        # 步驟 5：更新權重（教練依照步長修改神經元）
        optimizer.step()

        # 轉場小插曲：每跑 50 步，在進度條右邊顯示當前的 Loss
        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # ==================== 下半場：模擬考評估 (Evaluation) ====================
    # Evaluate your model
    matched = 0 #（總題數）   
    total = 0 #（答對題數）
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        '''
        for _, row in bar_eval: 裡的底線是忽略第一個值嗎？
            👉 完全正確！你的直覺 100% 命中！
            df_eval.iterrows() 是 Pandas 逐列走訪的工具。
            它每一輪一定會回傳一個長度為 2 的元組：(列編號 Index,該列的詳細資料 Series)
            但我們在考試時，根本不在乎這題是「第幾題（0, 1, 2...）」，我們只在乎裡面的題目與答案。
            所以用老朋友 _（單一底線） 當垃圾桶，把列編號丟掉，只把第二個有用的資料存給 row！
        '''
        batch_x = row['src'] # 拿到題目字串，例如 '1+1='
        batch_y = row['tgt'] # 拿到標準答案，例如 2

        # 步驟 1：模型使用我們寫的 generator 自回歸作答！
        prediction = model.generator(batch_x)# An example of using generator: model.generator('1+1=')
        '''
        考試與平時訓練的最大差別：
            平時訓練（上半場）有答案餵給模型（Teacher Forcing）；
            現在是模擬考！ 模型手裡只有題目（例如 batch_x = '1+1='），完全不知道答案。
        我們呼叫了剛才熱烈討論的 model.generator(batch_x)：
            模型啟動「自回歸生成」，一個字一個字心算，直到猜出 <eos> 停筆。
            回傳的 prediction 是一個字元清單：prediction=[’1’, ’+’, ’1’, ’=’, ’2’]
        '''

        # Write your code here. Input the batch_x to the model and generate the predictions
        # Write your code here.
        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total
        
        # 步驟 2：神級字串切片，只取出「等號後面的答案」
        pred_ans = ''.join(prediction[len(batch_x):])
        '''
        len(batch_x)：
            題目 '1+1=' 的長度是 4。
            prediction[4:]（切片語法）：
            從第 4 個位置開始往後切，剛好把前面的題目 '1+1=' 剁掉，只留下後面的答案 ['2']！
            ''.join(...)：
            用空字串把清單黏合起來，還原成純字串 '2'。
        大哉問：為什麼是 ''.join(...)？為什麼前面非要加一個點 .？
        這個問題問得太有深度了！這是所有從 C#、Java 轉到 Python 的人最常驚訝的經典語法冷知識！
            ① 為什麼不能直接呼叫 join(清單)？
                因為在 Python 裡，根本沒有一個獨立的全域函式叫 join()！
                Python 官方把 join 定義成 「字串物件（str）身上的方法（Method）」。
            ② 它的標準語法公式其實是這樣讀的：
                黏合膠水字串.join(要被黏起來的清單)
                看看這幾個生動的例子，你瞬間就秒懂了：
                words = ['蘋果', '香蕉', '橘子']
                # 例子 1：用「逗號」當膠水黏起來
                result = ','.join(words)
                # 產出："蘋果,香蕉,橘子"
            ③ 那為什麼我們寫 ''（空字串）？
                當我們的清單是字元 ['1', '2', '3'] 時： 我們希望它們緊密貼在一起變成 "123"，中間不要有任何空格或逗號！
                所以我們選用的「膠水」就是 ''（空字串 / 空氣）！
                而在 Python 裡，'' 本身就是一個合法的字串物件實體！
                在物件後面加個點 .，呼叫它身上的 join 方法：
                ''.join(['1', '2', '3'])
                # 產出："123"
                👉 所以那個點 .，就是物件導向的「調用方法」運算子！前面是字串膠水（空字串），後面是呼叫它的黏合功能！
        '''

        # 步驟 3：對答案（算準確率）
        if pred_ans == str(batch_y):
            matched += 1
        total += 1
        '''
        指標學名：Exact Match（完全匹配率，簡稱 EM）。
            評分原則（零容忍）：
            數學題沒有「同情分」。
            答案是 100，模型只要猜成 99 或 101，就算只差一點點，也是0 分！
            只有字元完全一模一樣（100% 吻合），matched 才會加 1。
        '''


    print(matched/total)

Train epoch 1: 100%|██████████| 15625/15625 [02:56<00:00, 88.46it/s, loss=1.27e-7]
Validation epoch 1: 10000it [32:57,  5.06it/s]


0.0


Train epoch 2: 100%|██████████| 15625/15625 [02:56<00:00, 88.30it/s, loss=1.7e-7]
Validation epoch 2: 1126it [03:43,  5.03it/s]

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/nlp-hw2/checkpoint.pt')
print('ok')

Running environment: colab

Python version: colab

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer:

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer:

##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer:

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer:

##### 5. Why do we need gradient clipping during training? (5%)

Answer:

##### … Anything that can strengthen your report. (5%)

Answer:

##### Generative AI Usage Statement

Answer:
